# 41 - Improved Experiments: Focal Loss + FER2013 Pre-Training

**Strategi baru untuk meningkatkan performa:**
1. **Focal Loss** (Lin et al., 2017) — fokus ke sampel sulit, ganti CrossEntropyLoss
2. **FER2013 Pre-Training** — ResNet18 pre-train di FER2013 (domain-specific TL)
3. **Kombinasi** — Focal Loss + FER2013 backbone

**Dataset:** Front-only 4-class (best config)
**Model:** CNN TL, Intermediate TL, Late Fusion TL
**Evaluasi:** Single Split (sama dengan eksperimen sebelumnya)

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
    EmotionCNNTransferFER, IntermediateFusionTransferFER,
)
from training.utils import (
    EmotionImageDataset, EmotionLandmarkDataset, EmotionMultimodalDataset,
    get_class_weights, train_model, full_evaluation,
    FocalLoss,
    plot_training_history, plot_confusion_matrix,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_frontonly_4class"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly" / "improved"
FER_WEIGHTS = PROJECT_ROOT / "models" / "pretrained" / "resnet18_fer2013.pth"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
NUM_CLASSES = 4
EMOTIONS = ["neutral", "happy", "sad", "negative"]

print(f"FER2013 weights: {FER_WEIGHTS.exists()}")
print("Setup complete.")

## Load Data

In [ ]:
from torch.utils.data import DataLoader

train_img_ds = EmotionImageDataset(DATASET_DIR / "X_train_images.npy", DATASET_DIR / "y_train.npy")
val_img_ds = EmotionImageDataset(DATASET_DIR / "X_val_images.npy", DATASET_DIR / "y_val.npy")
test_img_ds = EmotionImageDataset(DATASET_DIR / "X_test_images.npy", DATASET_DIR / "y_test.npy")

train_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_train_landmarks.npy", DATASET_DIR / "y_train.npy")
val_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_val_landmarks.npy", DATASET_DIR / "y_val.npy")
test_lm_ds = EmotionLandmarkDataset(DATASET_DIR / "X_test_landmarks.npy", DATASET_DIR / "y_test.npy")

train_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_train_images.npy", DATASET_DIR / "X_train_landmarks.npy", DATASET_DIR / "y_train.npy")
val_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_val_images.npy", DATASET_DIR / "X_val_landmarks.npy", DATASET_DIR / "y_val.npy")
test_mm_ds = EmotionMultimodalDataset(DATASET_DIR / "X_test_images.npy", DATASET_DIR / "X_test_landmarks.npy", DATASET_DIR / "y_test.npy")

train_img = DataLoader(train_img_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_img = DataLoader(val_img_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_img = DataLoader(test_img_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_lm = DataLoader(train_lm_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_lm = DataLoader(val_lm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_lm = DataLoader(test_lm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

train_mm = DataLoader(train_mm_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_mm = DataLoader(val_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_mm = DataLoader(test_mm_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

weights = get_class_weights(DATASET_DIR, device)
print(f"Train: {len(train_img_ds)} | Val: {len(val_img_ds)} | Test: {len(test_img_ds)}")
print(f"Class weights: {weights}")

## Helper

In [ ]:
all_results = {}

def run_experiment(name, model, criterion, train_loader, val_loader, test_loader,
                  model_type, lr=0.00005):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    save_path = str(OUTPUT_DIR / f"{name}.pth")
    
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    
    history, best_epoch = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                                       device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, model_type, EMOTIONS)
    
    print(f"  Acc={r['test_accuracy']:.4f} Macro-F1={r['test_macro_f1']:.4f}")
    all_results[name] = {
        "accuracy": float(r["test_accuracy"]),
        "macro_f1": float(r["test_macro_f1"]),
        "weighted_f1": float(r["test_weighted_f1"]),
    }
    return r

## Experiment 1: Focal Loss (ImageNet TL)

In [ ]:
# CNN TL + Focal Loss
model = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("CNN_TL_FocalLoss", model, criterion, train_img, val_img, test_img, "cnn")

# FCNN + Focal Loss
model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("FCNN_FocalLoss", model, criterion, train_lm, val_lm, test_lm, "fcnn", lr=0.0001)

# Intermediate TL + Focal Loss
model = IntermediateFusionTransfer(num_classes=NUM_CLASSES).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("IntermediateTL_FocalLoss", model, criterion, train_mm, val_mm, test_mm, "fusion")

## Experiment 2: FER2013 Pre-Training (CrossEntropyLoss)

In [ ]:
# CNN FER2013 TL + CrossEntropy
model = EmotionCNNTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
run_experiment("CNN_FER2013_CE", model, criterion, train_img, val_img, test_img, "cnn")

# Intermediate FER2013 TL + CrossEntropy
model = IntermediateFusionTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
run_experiment("IntermediateTL_FER2013_CE", model, criterion, train_mm, val_mm, test_mm, "fusion")

## Experiment 3: FER2013 + Focal Loss (Best Combo)

In [ ]:
# CNN FER2013 TL + Focal Loss
model = EmotionCNNTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("CNN_FER2013_Focal", model, criterion, train_img, val_img, test_img, "cnn")

# Intermediate FER2013 TL + Focal Loss
model = IntermediateFusionTransferFER(num_classes=NUM_CLASSES, fer_weights_path=str(FER_WEIGHTS)).to(device)
criterion = FocalLoss(gamma=2.0, alpha=weights)
run_experiment("IntermediateTL_FER2013_Focal", model, criterion, train_mm, val_mm, test_mm, "fusion")

## Late Fusion Variants

In [ ]:
# Late Fusion: CNN_FER2013_Focal + FCNN_FocalLoss
print("\nLate Fusion: FER2013 Focal + FCNN Focal")
cnn_model = EmotionCNNTransferFER(num_classes=NUM_CLASSES).to(device)
cnn_model.load_state_dict(torch.load(OUTPUT_DIR / "CNN_FER2013_Focal.pth", map_location=device, weights_only=True))
fcnn_model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
fcnn_model.load_state_dict(torch.load(OUTPUT_DIR / "FCNN_FocalLoss.pth", map_location=device, weights_only=True))
cnn_model.eval(); fcnn_model.eval()

cnn_probs_all, fcnn_probs_all, labels_all = [], [], []
with torch.no_grad():
    for images, landmarks, labels in test_mm:
        cnn_probs_all.append(torch.softmax(cnn_model(images.to(device)), dim=1).cpu().numpy())
        fcnn_probs_all.append(torch.softmax(fcnn_model(landmarks.to(device)), dim=1).cpu().numpy())
        labels_all.append(labels.numpy())

cnn_probs = np.concatenate(cnn_probs_all)
fcnn_probs = np.concatenate(fcnn_probs_all)
lbls = np.concatenate(labels_all)

best_f1, best_w = 0, 0.5
for w in np.arange(0.0, 1.05, 0.05):
    preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
    f1 = f1_score(lbls, preds, average="macro", zero_division=0)
    if f1 > best_f1: best_f1 = f1; best_w = w; best_preds = preds

acc = accuracy_score(lbls, best_preds)
wf1 = f1_score(lbls, best_preds, average="weighted", zero_division=0)
print(f"  LateFusion_FER2013_Focal: w={best_w:.2f} Acc={acc:.4f} F1={best_f1:.4f}")
all_results["LateFusion_FER2013_Focal"] = {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w}

## Ringkasan & Perbandingan

In [ ]:
print("="*75)
print("RINGKASAN IMPROVED EXPERIMENTS (4-class Front-Only)")
print("="*75)
print(f"{'Experiment':<35} {'Macro F1':>10} {'Accuracy':>10}")
print("-"*60)
for k in sorted(all_results.keys(), key=lambda x: -all_results[x]["macro_f1"]):
    r = all_results[k]
    print(f"{k:<35} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f}")

print("\nPerbandingan dengan baseline (sebelumnya):")
print(f"  Intermediate TL B1 (CE):       0.412")
print(f"  Late Fusion B3 (CE):           0.394")
print(f"  FCNN B3 (CE):                  0.361")

# Save
with open(OUTPUT_DIR / "improved_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR / 'improved_results.json'}")